In [3]:
import sys
import os
import json
from pathlib import Path

# 1. Path Setup
_PROJECT_ROOT = Path.cwd().parents[2]
if str(_PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(_PROJECT_ROOT))

_RQ1_DIR = Path.cwd().parent
if str(_RQ1_DIR) not in sys.path:
    sys.path.insert(0, str(_RQ1_DIR))

from evaluate_rq1 import *
print(f"✓ Success! Project root set to: {_PROJECT_ROOT}")

# 2. Define File Paths
# Point this directly to your new DBpedia test file
TEST_DATA_PATH = os.path.join(_PROJECT_ROOT, "datasets", "dbpedia_testset_RQ1.json")
CHECKPOINT_PATH = os.path.join(_PROJECT_ROOT, "checkpoints", "sparqlce_dbpedia", "best_model.pt")

TOPKS = [3, 5, 10, 20]
BATCH = 32

# 3. Load the test dataset directly
test_data = load_triplets(TEST_DATA_PATH)

print(f"Dataset Loaded -> Test Split: {len(test_data)}")

# 4. Build corpus and eval items using ONLY the test data
global_corpus, seen_q = build_global_corpus(test_data)
eval_items = build_eval_items(test_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Evaluation Ready -> Anchors: {len(eval_items)} | Corpus: {len(global_corpus)}")
# Enable per-bin lexical stratification for every model below:
#   [0,0.3) adversarial | [0.3,0.5) divergent | [0.5,0.8) moderate | [0.8,1.0) near-identical
import evaluate_rq1
evaluate_rq1.STRATIFY_BINS = [0.3, 0.5, 0.8]  # cumulative: cos < threshold


✓ Success! Project root set to: /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings
Dataset Loaded -> Test Split: 462
Evaluation Ready -> Anchors: 462 | Corpus: 1841


In [4]:
# SPARQLCE
evaluate_sparqlce_checkpoint(CHECKPOINT_PATH, eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

# TF-IDF
evaluate_tfidf(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# BM25
evaluate_bm25(eval_items, global_corpus, relevance_scores_per_item, TOPKS)

# E5
evaluate_sentence_transformer('intfloat/e5-base-v2', 'E5-base-v2', eval_items, global_corpus, relevance_scores_per_item, TOPKS, transform=lambda q: f'query: {q}')

# Qwen2.5-Coder (generative code LLM used as an embedder, last-token pooling;
# 0.5B ≈ scale-matched to the dense 0.6B Qwen3-Embedding it replaces)
evaluate_code_llm('Qwen/Qwen2.5-Coder-0.5B', 'Qwen2.5-Coder-0.5B', eval_items, global_corpus, relevance_scores_per_item, TOPKS)

Loading model from /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/checkpoints/sparqlce_dbpedia/best_model.pt...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 981.55it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



SPARQLCE
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.4690  0.4505  0.3429  0.7233  0.7727    0.4690  0.6689
@5     0.3961  0.5116  0.3247  0.7337  0.8182    0.3961  0.6712
@10    0.3214  0.6122  0.3158  0.7373  0.8442    0.3214  0.6787
@20    0.2479  0.7435  0.2990  0.7404  0.8896    0.2479  0.6915
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.2763   0.2930   0.3087   0.3197
            <0.5    0.2787      0.2921      187   0.5980   0.6079   0.6161   0.6231
            <0.8    0.4967      0.5867      376   0.7416   0.7533   0.7571   0.7601

Baseline: TF-IDF
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
------------------------

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1275.82it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: intfloat/e5-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



E5-base-v2
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1970  0.2782  0.1842  0.3831  0.4394    0.1970  0.3354
@5     0.1468  0.2976  0.1513  0.3911  0.4740    0.1468  0.3332
@10    0.1035  0.3315  0.1205  0.3997  0.5390    0.1035  0.3358
@20    0.0751  0.3763  0.0998  0.4063  0.6342    0.0751  0.3426
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0468   0.0468   0.0504   0.0566
            <0.5    0.2787      0.2921      187   0.1399   0.1490   0.1552   0.1612
            <0.8    0.4967      0.5867      376   0.2407   0.2511   0.2615   0.2697


`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 840.55it/s, Materializing param=norm.weight]                              



Qwen2.5-Coder-0.5B
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1378  0.1483  0.1074  0.2478  0.2987    0.1378  0.1995
@5     0.1165  0.1757  0.1020  0.2588  0.3463    0.1165  0.2037
@10    0.0868  0.2236  0.0923  0.2715  0.4437    0.0868  0.2094
@20    0.0626  0.2754  0.0802  0.2784  0.5433    0.0626  0.2144
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0439   0.0461   0.0475   0.0492
            <0.5    0.2787      0.2921      187   0.1168   0.1213   0.1271   0.1310
            <0.8    0.4967      0.5867      376   0.2057   0.2179   0.2305   0.2376


In [5]:
# Prior-literature SPARQL / code encoder baselines (zero-shot, mean-pooled):
#   CodeBERT (SparqlCE's own backbone), GraphCodeBERT (CodeBERT + data-flow
#   graph), SPBERT (BERT pretrained on SPARQL). Defined in evaluate_rq1.py as
#   SPARQL_ENCODER_BASELINES. The "mean pooling" notice each emits is correct
#   here (raw MLM encoders), not the LoRA-adapter bug.
evaluate_sparql_encoder_baselines(eval_items, global_corpus, relevance_scores_per_item, TOPKS, batch_size=BATCH)

No sentence-transformers model found with name microsoft/codebert-base. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1066.94it/s, Materializing param=pooler.dense.weight]                               



CodeBERT (zero-shot)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1623  0.1887  0.1302  0.2983  0.3528    0.1623  0.2516
@5     0.1286  0.2131  0.1173  0.3112  0.4069    0.1286  0.2503
@10    0.0916  0.2417  0.0983  0.3201  0.4719    0.0916  0.2476
@20    0.0680  0.2870  0.0868  0.3250  0.5433    0.0680  0.2528
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0439   0.0439   0.0439   0.0469
            <0.5    0.2787      0.2921      187   0.1497   0.1575   0.1629   0.1663
            <0.8    0.4967      0.5867      376   0.2114   0.2273   0.2379   0.2436


No sentence-transformers model found with name microsoft/graphcodebert-base. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1309.90it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: microsoft/graphcodebert-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.decoder.weight          | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.decoder.bias            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect ide


GraphCodeBERT (zero-shot)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1688  0.2020  0.1385  0.3081  0.3680    0.1688  0.2622
@5     0.1320  0.2209  0.1207  0.3167  0.4048    0.1320  0.2589
@10    0.0905  0.2493  0.0983  0.3240  0.4589    0.0905  0.2546
@20    0.0649  0.2987  0.0838  0.3290  0.5303    0.0649  0.2590
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0395   0.0395   0.0443   0.0466
            <0.5    0.2787      0.2921      187   0.1711   0.1746   0.1800   0.1839
            <0.8    0.4967      0.5867      376   0.2332   0.2437   0.2526   0.2579


No sentence-transformers model found with name razent/spbert-mlm-wso-base. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1481.57it/s, Materializing param=encoder.layer.11.output.dense.weight]              
BertModel LOAD REPORT from: razent/spbert-mlm-wso-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                       


SPBERT (zero-shot)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1659  0.1761  0.1283  0.2821  0.3550    0.1659  0.2379
@5     0.1355  0.2056  0.1188  0.2934  0.4048    0.1355  0.2408
@10    0.1024  0.2539  0.1072  0.3068  0.5022    0.1024  0.2468
@20    0.0768  0.3129  0.0971  0.3121  0.5801    0.0768  0.2545
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0395   0.0395   0.0407   0.0428
            <0.5    0.2787      0.2921      187   0.1355   0.1403   0.1502   0.1532
            <0.8    0.4967      0.5867      376   0.2340   0.2452   0.2594   0.2651


In [6]:

display_name = "SBERT/MiniLM" 
model_id = "sentence-transformers/all-MiniLM-L6-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 903.59it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



SBERT/MiniLM
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1861  0.2423  0.1632  0.3514  0.4199    0.1861  0.2994
@5     0.1433  0.2745  0.1426  0.3596  0.4567    0.1433  0.3004
@10    0.0994  0.3132  0.1142  0.3674  0.5130    0.0994  0.3018
@20    0.0746  0.3705  0.0984  0.3743  0.6061    0.0746  0.3119
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0629   0.0646   0.0670   0.0698
            <0.5    0.2787      0.2921      187   0.1533   0.1602   0.1653   0.1697
            <0.8    0.4967      0.5867      376   0.2451   0.2538   0.2623   0.2697


In [7]:
display_name = "SBERT/MPNet"
model_id = "sentence-transformers/all-mpnet-base-v2"
evaluate_sentence_transformer(model_id, display_name, eval_items, global_corpus, relevance_scores_per_item, TOPKS)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 869.28it/s, Materializing param=pooler.dense.weight]                        
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



SBERT/MPNet
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.1898  0.2555  0.1704  0.3723  0.4329    0.1898  0.3180
@5     0.1606  0.2884  0.1559  0.3906  0.5130    0.1606  0.3271
@10    0.1134  0.3295  0.1272  0.3982  0.5693    0.1134  0.3294
@20    0.0785  0.3763  0.1031  0.4021  0.6277    0.0785  0.3332
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.0497   0.0550   0.0574   0.0615
            <0.5    0.2787      0.2921      187   0.1453   0.1592   0.1647   0.1675
            <0.8    0.4967      0.5867      376   0.2478   0.2704   0.2789   0.2835


In [8]:
# E5-base-v2 (fine-tuned on SQuARE-DBpedia) — capacity-matched trained baseline
# Checkpoint saved by 04_e5_contrastive_finetune_square.ipynb
E5_FT_PATH = os.path.join(_PROJECT_ROOT, "checkpoints", "e5_square_dbpedia", "best_model")
evaluate_sentence_transformer(
    E5_FT_PATH,
    "E5-base-v2 (fine-tuned)",
    eval_items,
    global_corpus,
    relevance_scores_per_item,
    TOPKS,
    transform=lambda q: f"query: {q}",
)

No sentence-transformers model found with name /local/upb/users/z/zahera/profiles/unix/cs/SPARQL-Embeddings/checkpoints/e5_square_dbpedia/best_model. Creating a new one with mean pooling.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 920.92it/s, Materializing param=pooler.dense.weight]                               



E5-base-v2 (fine-tuned)
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.3773  0.4503  0.3113  0.6497  0.7359    0.3773  0.5928
@5     0.3117  0.4997  0.2795  0.6637  0.7965    0.3117  0.6007
@10    0.2442  0.5794  0.2520  0.6716  0.8615    0.2442  0.6103
@20    0.1896  0.6811  0.2336  0.6749  0.9069    0.1896  0.6236
  -- stratified by anchor-positive lexical similarity (TF-IDF cosine, cumulative) --
     lexical bin  mean_cos  median_cos  anchors MRR@3    MRR@5    MRR@10   MRR@20  
            <0.3    0.1847      0.1953      114   0.1915   0.2016   0.2118   0.2191
            <0.5    0.2787      0.2921      187   0.4545   0.4679   0.4763   0.4808
            <0.8    0.4967      0.5867      376   0.5638   0.5816   0.5914   0.5954
